<a href="https://colab.research.google.com/github/Airplane356/video-background-remover/blob/main/SAM3_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e ".[notebooks]"
!pip install triton
!pip install einops ninja && pip install flash-attn-3 --no-deps --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Authenticate Hugging Face
from google.colab import userdata
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from sam3.model_builder import build_sam3_video_predictor

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Config
INPUT_VIDEO = "/content/video.mp4"
OUTPUT_VIDEO = "/content/output_nobg.mp4"
TEMP_VIDEO = "/content/_sam_input.mp4"
TEXT_PROMPT = "dogs"
BG_COLOR = [0, 255, 0] # green screen
SAM_MAX_SIDE = 720

# Load model
try:
    # clear memory
    del video_predictor
    gc.collect()
    torch.cuda.empty_cache()
except NameError:
    pass

video_predictor = build_sam3_video_predictor()

# Read vid metadata
vid_object = cv2.VideoCapture(INPUT_VIDEO)
fps = vid_object.get(cv2.CAP_PROP_FPS)
W = int(vid_object.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(vid_object.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(vid_object.get(cv2.CAP_PROP_FRAME_COUNT))

# Set to frame 0
vid_object.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Downscale for SAM3 (keeps VRAM low for T4)
scale = min(1.0, SAM_MAX_SIDE / max(W, H))
SW = int(W * scale) & ~1
SH = int(H * scale) & ~1

downscaled_vid = cv2.VideoWriter(TEMP_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (SW, SH))
for i in tqdm(range(n_frames), desc=f"Downscaling to {SW}×{SH}"):
    retval, frame = vid_object.read()
    if not retval:
        break
    downscaled_vid.write(cv2.resize(frame, (SW, SH)))

vid_object.release()
downscaled_vid.release()

# SAM3 session + prompt
torch.cuda.empty_cache()

response = video_predictor.handle_request({
    "type": "start_session",
    "resource_path": TEMP_VIDEO,
    "offload_video_to_cpu": True,
    "offload_state_to_cpu": True
})

session_id = response["session_id"]

with torch.inference_mode():
    response = video_predictor.handle_request({
        "type": "add_prompt",
        "session_id": session_id,
        "frame_index": 0,
        "text": TEXT_PROMPT
    })

# Propagate masks
all_masks = {}

with torch.inference_mode():
    stream = video_predictor.handle_stream_request({
        "type": "propagate_in_video",
        "session_id": session_id
    })
    for frame_data in tqdm(stream, total=n_frames, desc="Propagating"):
        idx = frame_data["frame_index"]
        outputs = frame_data.get("outputs")
        if not outputs or not len(outputs.get("out_obj_ids", [])):
            continue

        combined = np.zeros((SH, SW), dtype=bool)
        for mask in outputs["out_binary_masks"]:
            if isinstance(mask, torch.Tensor):
                mask = mask.cpu().numpy()
            combined |= mask.astype(bool)

        # upscale mask back to original resolution with gaussian blur
        mask_upscaled = cv2.resize(combined.astype(np.uint8), (W, H), interpolation=cv2.INTER_LINEAR)
        mask_smooth   = cv2.GaussianBlur(mask_upscaled.astype(np.float32), (21, 21), 0)
        all_masks[idx] = mask_smooth > 0.5

video_predictor.handle_request({
    "type": "close_session",
    "session_id": session_id,
    "run_gc_collect": True
})

# Write output
vid_object = cv2.VideoCapture(INPUT_VIDEO)
final_vid = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

for frame_idx in tqdm(range(n_frames), desc="Writing"):
    retval, frame = vid_object.read()
    if not retval:
        break
    mask = all_masks.get(frame_idx)
    if mask is not None:
        frame[~mask] = BG_COLOR
    final_vid.write(frame)

vid_object.release()
final_vid.release()
print("DONE REMOVING BACKGROUND")